In [1]:
## Define Sentiment Labels (TRAINING ONLY)
def label_sentiment(rating):
    if rating >= 9.0:
        return "Positive"
    elif rating >= 7.0:
        return "Neutral"
    else:
        return "Negative"

In [2]:
## Prepare GLOBAL Training Data
import pandas as pd

def prepare_global_training_data(csv_path):
    df = pd.read_csv(csv_path)

    # Select & clean columns
    df = df[["IMDB_Rating", "Released_Year"]].dropna()

    df.rename(columns={
        "IMDB_Rating": "rating",
        "Released_Year": "year"
    }, inplace=True)

    df["rating"] = df["rating"].astype(float)
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df.dropna()

    # Sort by year to simulate time progression
    df = df.sort_values("year")

    # Feature engineering
    df["rating_diff"] = df["rating"].diff().fillna(0)

    # Labels
    df["sentiment"] = df["rating"].apply(label_sentiment)

    return df


In [3]:
## Train GLOBAL ML Model (ONCE)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

def train_global_model(df):
    X = df[["rating", "rating_diff", "year"]]
    y = df["sentiment"]

    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(y)

    model = LogisticRegression(max_iter=1000)
    model.fit(X, y_encoded)

    return model, encoder

In [4]:
## Extract Actor / Director Last 5 Movies (From Kaggle)
def get_person_last_movies(df, name):
    person_df = df[
        (df["Director"] == name) |
        (df["Star1"] == name)
    ][["Released_Year", "IMDB_Rating"]].dropna()

    person_df = person_df.rename(columns={
        "Released_Year": "year",
        "IMDB_Rating": "rating"
    })

    # 🔥 CRITICAL FIX
    person_df["year"] = pd.to_numeric(person_df["year"], errors="coerce")
    person_df["rating"] = person_df["rating"].astype(float)

    person_df = person_df.dropna()

    person_df = person_df.sort_values("year").tail(5)

    if len(person_df) < 3:
        raise ValueError("Not enough movies for prediction")

    person_df["rating_diff"] = person_df["rating"].diff().fillna(0)

    return person_df



In [5]:
## Predict NEXT Movie Sentiment
def predict_next_movie(model, encoder, person_df):
    last = person_df.iloc[-1]

    X_next = [[
        last["rating"],
        last["rating_diff"],
        last["year"] + 1
    ]]

    prediction = model.predict(X_next)
    confidence = model.predict_proba(X_next).max()

    return encoder.inverse_transform(prediction)[0], confidence


In [6]:
## MAIN APPLICATION
def run_app():
    print("\n🎬 IMDB NEXT MOVIE SENTIMENT PREDICTOR\n")

    name = input("Enter Actor or Director Name: ")
    file_path = "../files/imdb_top_1000.csv"
    # Load & train global model
    global_df = prepare_global_training_data(file_path)
    model, encoder = train_global_model(global_df)

    # Reload full dataset for person filtering
    full_df = pd.read_csv(file_path)

    person_df = get_person_last_movies(full_df, name)

    sentiment, confidence = predict_next_movie(model, encoder, person_df)

    print("\n📊 Last Movies Used:")
    print(person_df)

    print("\n🎯 Predicted NEXT Movie Sentiment:")
    print(f"👉 {sentiment} (confidence: {confidence:.2f})")


In [7]:
## Run
if __name__ == "__main__":
    run_app()


🎬 IMDB NEXT MOVIE SENTIMENT PREDICTOR



ValueError: Not enough movies for prediction